# Chemical vs Embedding Validation
Este notebook valida se o espaço latente gerado pelos embeddings do modelo de soldagem reflete o comportamento real da composição química dos materiais.

In [1]:
#Importações e Configurações Iniciais
import pandas as pd
import numpy as np

In [ ]:
import onnx, pandas as pd
from onnx import numpy_helper

BASE = r'C:\Users\Workstation\Documents\GitHub\welding-optuna-pytorch'

model = onnx.load(BASE + r'\melhor_modelo_producao.onnx')
emb = {i.name: numpy_helper.to_array(i) for i in model.graph.initializer}['material_encoder.embedding.weight']

materiais = [
    "316", "304", "duplex 2205",
    "A106 GrA", "A106 GrB", "A106 GrB",
    "A333 Gr6", 
    "A333 Gr8"
]

df = pd.DataFrame(emb[:len(materiais)], columns=[f"dim_{i}" for i in range(emb.shape[1])])
df.insert(0, "material", materiais)
df.to_csv(BASE + r'\embeddings_filtered_token.csv', index=False)
print(df)

       material     dim_0     dim_1     dim_2     dim_3     dim_4     dim_5  \
0           316 -0.695846 -0.880078 -0.003047 -1.594203 -0.582159 -0.396439   
1           304  0.109703  0.473873 -1.600065 -1.691428 -0.701978 -0.220523   
2   duplex 2205  0.813232  1.313394 -0.243713  0.376286 -0.127266  0.993751   
3      A106 GrA  2.849302 -1.008900 -1.591877 -1.330585  0.023337 -0.296782   
4      A106 GrB  0.628136 -1.420547 -1.153010  0.104143 -0.107090  0.888972   
5      A106 GrB -0.983748  0.063789  0.042676  0.112477  0.087193 -2.662720   
6      A333 Gr1 -0.604981  0.651536 -0.739579  0.340141  0.041759  0.646432   
7      A333 Gr3  1.300862  0.046789  0.326452  0.289517  0.197170  0.093269   
8      A333 Gr4  1.137535 -1.485973  0.056058  0.706873  1.840266 -1.241353   
9      A333 Gr6 -1.657265 -0.332178  1.163174 -2.436434 -0.926246 -0.375850   
10     A333 Gr7  1.205286  1.097787  1.233218  0.889398  0.103967 -0.829960   
11     A333 Gr8  0.859901  0.941691 -0.456104 -0.668

In [9]:
#Funções Auxiliares
def calcular_overlap(intervalo_a, intervalo_b):
    if not intervalo_a or not intervalo_b:
        return 0
    maior_inicio = max(intervalo_a[0], intervalo_b[0])
    menor_fim = min(intervalo_a[1], intervalo_b[1])
    return max(0, menor_fim - maior_inicio)

def extrair_intervalo(valor):
    if pd.isna(valor) or valor == '-':
        return None
    
    valor_str = str(valor).replace(',', '.').strip()
    
    try:
        if '≤' in valor_str:
            num = float(valor_str.replace('≤', '').strip())
            return (0.0, num) 
        elif '≥' in valor_str:
            num = float(valor_str.replace('≥', '').strip())
            return (num, float('inf'))
        elif '-' in valor_str:
            partes = valor_str.split('-')
            return (float(partes[0].strip()), float(partes[1].strip()))
        else:
            num = float(valor_str)
            return (num, num)
    except:
        return None

def calcular_similaridade_percentual(int_a, int_b):
    if not int_a or not int_b: return 0.0
    if int_a == int_b: return 1.0 
        
    inicio_inter = max(int_a[0], int_b[0])
    fim_inter = min(int_a[1], int_b[1])
    intersecao = max(0.0, fim_inter - inicio_inter)
    
    if intersecao == 0: return 0.0
        
    inicio_uniao = min(int_a[0], int_b[0])
    fim_uniao = max(int_a[1], int_b[1])
    uniao = max(0.0001, fim_uniao - inicio_uniao) 
    
    return intersecao / uniao

def calcular_similaridade_cosseno(vetor_a, vetor_b):
    vetor_a = np.array(vetor_a, dtype=float)
    vetor_b = np.array(vetor_b, dtype=float)

    produto_escalar = np.dot(vetor_a, vetor_b)
    norma_a = np.linalg.norm(vetor_a)
    norma_b = np.linalg.norm(vetor_b)

    if norma_a == 0 or norma_b == 0:
        return 0.0

    return produto_escalar / (norma_a * norma_b)

def calcular_distancia_euclidiana(vetor_a, vetor_b):
    vetor_a = np.array(vetor_a, dtype=float)
    vetor_b = np.array(vetor_b, dtype=float)

    return np.sqrt(np.sum((vetor_a - vetor_b) ** 2))

In [10]:
# Carregar dados químicos
df_quimica = pd.read_excel(r'C:\Users\Workstation\Documents\GitHub\welding-optuna-pytorch\data\material_apoio\comp test.xlsx')
df_quimica.rename(columns={"Unnamed: 0": "material"}, inplace=True)

elementos = ['C', 'Si', 'Mn', 'P', 'S', 'N', 'Cr', 'Mo', 'Cu', 'V', 'Al', 'Ni', 'Ti', 'Co']
dados_limpo = df_quimica[["material"] + elementos].copy()

for col in elementos:
    dados_limpo[col + '_int'] = dados_limpo[col].apply(extrair_intervalo)

materiais_q = dados_limpo['material'].tolist()
matriz_quimica = pd.DataFrame(index=materiais_q, columns=materiais_q, dtype=float)

for i in range(len(materiais_q)):
    for j in range(len(materiais_q)):
        valores_similaridade = []
        for col in elementos:
            int_a = dados_limpo.loc[i, col + '_int']
            int_b = dados_limpo.loc[j, col + '_int']
            if int_a is not None and int_b is not None:
                valores_similaridade.append(calcular_similaridade_percentual(int_a, int_b))
        matriz_quimica.iloc[i, j] = round(np.mean(valores_similaridade), 3) if valores_similaridade else 0.0

print("=== Matriz de Similaridade Química (Composição) ===")
display(matriz_quimica)

=== Matriz de Similaridade Química (Composição) ===


,316,304,duplex 2205,A106 GrA,A106 GrB,A106 GrB,A333 Gr1,A333 Gr3,A333 Gr4,A333 Gr6,A333 Gr7,A333 Gr8,A333 Gr9,A333 Gr10,A333 Gr11
316,1.000,0.804,0.399,0.281,0.282,0.277,0.082,0.059,0.081,0.048,0.107,0.107,0.066,0.053,0.081
304,0.804,1.000,0.456,0.321,0.322,0.317,0.082,0.059,0.081,0.055,0.107,0.187,0.066,0.061,0.093
duplex 2205,0.399,0.456,1.000,0.187,0.192,0.190,0.110,0.071,0.094,0.055,0.128,0.128,0.082,0.061,0.093
A106 GrA,0.281,0.321,0.187,1.000,0.964,0.952,0.168,0.083,0.069,0.456,0.113,0.113,0.112,0.138,0.182
A106 GrB,0.282,0.322,0.192,0.964,1.000,0.986,0.214,0.071,0.089,0.475,0.096,0.096,0.143,0.138,0.174
A106 GrB,0.277,0.317,0.190,0.952,0.986,1.000,0.214,0.071,0.089,0.475,0.096,0.096,0.143,0.138,0.174
A333 Gr1,0.082,0.082,0.110,0.168,0.214,0.214,1.000,0.580,0.708,0.964,0.618,0.618,0.750,0.000,0.547
A333 Gr3,0.059,0.059,0.071,0.083,0.071,0.071,0.580,1.000,0.474,0.405,0.658,0.492,0.464,0.105,0.485
A333 Gr4,0.081,0.081,0.094,0.069,0.089,0.089,0.708,0.474,1.000,0.339,0.506,0.506,0.472,0.103,0.412
A333 Gr6,0.048,0.055,0.055,0.456,0.475,0.475,0.964,0.405,0.339,1.000,0.429,0.429,0.476,0.150,0.362


In [5]:
# Carregar embeddings (um CSV por modelo treinado: coluna "material" + colunas do vetor)
import os

BASE = r'C:\Users\Workstation\Documents\GitHub\welding-optuna-pytorch'
EMBEDDINGS = {
    "token":       BASE + r'\embeddings_filtered_token.csv',
}

def carregar_embeddings(caminho):
    df_emb = pd.read_csv(caminho)
    materiais_e = df_emb["material"].tolist()
    colunas_vetor = [c for c in df_emb.columns if c != "material"]
    return df_emb, materiais_e, colunas_vetor

In [11]:
# Matrizes de cosseno e euclidiana para cada conjunto de embeddings
resultados = {}

for nome, caminho in EMBEDDINGS.items():
    if not os.path.exists(caminho):
        print(f"[{nome}] arquivo não encontrado, pulando: {caminho}")
        continue

    df_emb, materiais_e, colunas_vetor = carregar_embeddings(caminho)

    matriz_cosseno = pd.DataFrame(index=materiais_e, columns=materiais_e, dtype=float)

    for i in range(len(materiais_e)):
        for j in range(len(materiais_e)):
            v_a = df_emb.loc[i, colunas_vetor]
            v_b = df_emb.loc[j, colunas_vetor]
            matriz_cosseno.iloc[i, j] = round(calcular_similaridade_cosseno(v_a, v_b), 3)


    print(f"=== [{nome}] Matriz de Similaridade por Cosseno (Embeddings) ===")
    display(matriz_cosseno)


=== [token] Matriz de Similaridade por Cosseno (Embeddings) ===


,316,304,duplex 2205,A106 GrA,A106 GrB,A106 GrB,A333 Gr1,A333 Gr3,A333 Gr4,A333 Gr6,A333 Gr7,A333 Gr8,A333 Gr9,A333 Gr10,A333 Gr11
316,1.000,0.506,-0.459,0.104,0.033,0.281,-0.095,-0.508,-0.205,0.841,-0.601,-0.134,0.307,-0.217,0.821
304,0.506,1.000,0.060,0.459,0.171,0.270,0.239,-0.362,-0.367,0.339,-0.387,0.116,0.022,0.240,0.160
duplex 2205,-0.459,0.060,1.000,-0.159,-0.269,-0.319,0.698,-0.057,-0.367,-0.135,0.172,0.410,-0.411,0.064,-0.317
A106 GrA,0.104,0.459,-0.159,1.000,0.656,-0.178,-0.458,0.546,0.370,-0.324,0.010,0.265,0.050,0.171,-0.041
A106 GrB,0.033,0.171,-0.269,0.656,1.000,-0.427,-0.255,0.313,0.248,-0.425,-0.432,-0.285,0.343,0.290,0.073
A106 GrB,0.281,0.270,-0.319,-0.178,-0.427,1.000,0.102,-0.497,0.186,0.335,0.132,-0.218,-0.187,0.262,0.172
A333 Gr1,-0.095,0.239,0.698,-0.458,-0.255,0.102,1.000,-0.705,-0.409,0.186,-0.349,0.011,-0.312,0.374,0.037
A333 Gr3,-0.508,-0.362,-0.057,0.546,0.313,-0.497,-0.705,1.000,0.410,-0.676,0.599,0.298,0.019,-0.208,-0.495
A333 Gr4,-0.205,-0.367,-0.367,0.370,0.248,0.186,-0.409,0.410,1.000,-0.431,0.243,0.057,-0.123,0.404,-0.039
A333 Gr6,0.841,0.339,-0.135,-0.324,-0.425,0.335,0.186,-0.676,-0.431,1.000,-0.391,-0.121,0.261,-0.288,0.669


A análise comparativa entre as matrizes de similaridade química e o espaço latente dos embeddings permite concluir que o notebook não valida a hipótese de que a rede neural reflete o comportamento real da composição química dos materiais. Enquanto a composição química agrupa ligas semelhantes de forma lógica,como os aços inoxidáveis 316 e 304, o espaço vetorial do modelo gerou correlações divergentes.